In [2]:
import pandas as pd

# Load the cleaned CSV
df = pd.read_csv("merlin_meta_german_clean.csv")

# Check the first few rows
print(df.head())


                                                text CEFR
0  M. Meier Müllergasse 1 Stadt X Internationale ...   B2
1  Müller Julia Bahnhofsstr. 1 A Stadt X Armenien...   B2
2  Michael Meier 1 Zentralplatz 1234. Stadt X Aup...   B2
3  Eva Meier Schmidt Müllergasse 12 Stadt X Kroat...   B2
4  Abs. Frau EVA SCHMIDT BAHNHOFSTR, , 1234 STADT...   B1


## Logistic Regression

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Load CSV
df = pd.read_csv("merlin_meta_german_clean.csv")

# Step 2: Check class distribution
print("Class distribution:\n", df['CEFR'].value_counts())

# Step 3: Split into train/test (stratified to preserve class proportions)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['CEFR'], test_size=0.2, random_state=42, stratify=df['CEFR']
)

# Step 4: Convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 5: Train Logistic Regression with class weights
lr = LogisticRegression(
    max_iter=1000,
    multi_class='multinomial',
    solver='lbfgs',
    class_weight='balanced'   # <-- this balances minority classes
)
lr.fit(X_train_tfidf, y_train)

# Step 6: Predict and evaluate
y_pred = lr.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Class distribution:
 CEFR
B1    331
A2    306
B2    293
A1     57
C1     42
C2      4
Name: count, dtype: int64


D:\anaconda\envs\tf_env\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.6473429951690821

Classification Report:
               precision    recall  f1-score   support

          A1       0.64      0.58      0.61        12
          A2       0.74      0.61      0.67        61
          B1       0.61      0.65      0.63        66
          B2       0.70      0.73      0.72        59
          C1       0.29      0.50      0.36         8
          C2       0.00      0.00      0.00         1

    accuracy                           0.65       207
   macro avg       0.50      0.51      0.50       207
weighted avg       0.66      0.65      0.65       207



D:\anaconda\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\anaconda\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\anaconda\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [6]:
# Separate minority classes
minority_df = df[df['CEFR'].isin(['A1', 'C1', 'C2'])]
majority_df = df[~df['CEFR'].isin(['A1', 'C1', 'C2'])]
